# Fixlane — Data Exploration

Run this before training to understand the dataset.
No GPU needed — runs in a few seconds on regular CPU.

In [ ]:
import json
from collections import Counter

# Simple helper to load a .jsonl file
def load_jsonl(file_path):
    rows = []
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_data = load_jsonl("./data/train.jsonl")
val_data   = load_jsonl("./data/val.jsonl")
test_data  = load_jsonl("./data/test.jsonl")

print(f"Train: {len(train_data)} examples")
print(f"Val:   {len(val_data)} examples")
print(f"Test:  {len(test_data)} examples")

In [ ]:
# Count how often each vehicle system and severity label appears
system_counts   = Counter()
severity_counts = Counter()

for row in train_data:
    system_counts[row["output"]["vehicle_system"]] += 1
    severity_counts[row["output"]["severity"]] += 1

print("--- Vehicle System Distribution (train) ---")
for system, count in system_counts.most_common():
    pct = count / len(train_data) * 100
    print(f"  {system:<15}: {count:>4}  ({pct:.1f}%)")

print()
print("--- Severity Distribution (train) ---")
for severity, count in severity_counts.most_common():
    pct = count / len(train_data) * 100
    print(f"  {severity:<20}: {count:>4}  ({pct:.1f}%)")

In [ ]:
# Check input length in words
lengths = [len(row["input"].split()) for row in train_data]
lengths_sorted = sorted(lengths)
n = len(lengths_sorted)

print("--- Input Length (words) ---")
print(f"  Min:    {lengths_sorted[0]}")
print(f"  Max:    {lengths_sorted[-1]}")
print(f"  Mean:   {sum(lengths) / n:.1f}")
print(f"  Median: {lengths_sorted[n // 2]}")

In [ ]:
# Look for duplicate inputs
# (The dataset has some inputs repeated with minor wording changes)
seen = {}
dupe_groups = 0

for row in train_data:
    normalized = row["input"].lower().strip()
    if normalized in seen:
        if seen[normalized] == 1:
            dupe_groups += 1
        seen[normalized] += 1
    else:
        seen[normalized] = 1

print(f"Exact-duplicate input groups: {dupe_groups}")
print("(Near-paraphrase duplicates exist too — these are intentional dataset noise)")

In [ ]:
# Flag suspicious severity labels
# Brakes labeled as 'comfort' is the main thing to call out
print("Suspicious rows (brakes + comfort):")
count = 0
for row in train_data:
    sys = row["output"]["vehicle_system"]
    sev = row["output"]["severity"]
    if sys == "brakes" and sev in ("comfort", "cosmetic"):
        print(f'  "{row["input"]}"')
        count += 1

print(f"\nTotal: {count} rows")
print("These are morning brake squeak cases — probably not mislabeled, just ambiguous.")

In [ ]:
# Print a few sample rows to get a feel for the data
print("--- Sample Training Rows ---")
for row in train_data[:5]:
    print(f"  Input:  {row['input']}")
    print(f"  Output: {json.dumps(row['output'])}")
    print()